# 🚗 Lab Complet : Analyse des Séries Temporelles — Tarification Assurance Auto

**Objectif :** Analyser l'évolution mensuelle de la **prime pure** d'un portefeuille d'assurance automobile et produire des prévisions via plusieurs modèles de séries temporelles.

**Modèles appliqués :**
- Tests de stationnarité (ADF, KPSS)
- ACF / PACF
- AR, MA, ARMA, ARIMA, SARIMA
- Prophet (Facebook)
- Analyse complète des résidus
- Comparaison des modèles et conclusions

---

## 📦 Étape 0 — Installation des bibliothèques

In [ ]:
# Installation des bibliothèques nécessaires
!pip install prophet statsmodels pmdarima --quiet

## 📚 Étape 1 — Importation des bibliothèques

In [ ]:
# ── Données & calcul ────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Visualisation ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# ── Séries temporelles ──────────────────────────────────────────────────────
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats
import pmdarima as pm   # auto_arima
from prophet import Prophet

# ── Métriques ───────────────────────────────────────────────────────────────
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("✅ Toutes les bibliothèques ont été importées avec succès.")

## 📂 Étape 2 — Chargement et Prétraitement des Données

> **Note Colab :** Chargez votre fichier `Motor_vehicle_insurance_data.csv` via le bouton 📁 dans le panneau gauche, ou exécutez la cellule suivante pour le charger depuis Google Drive.

In [ ]:
# ── Option 1 : Upload direct dans Colab ─────────────────────────────────────
# from google.colab import files
# uploaded = files.upload()   # décommentez pour un upload interactif

# ── Option 2 : Chemin local (remplacez par votre chemin réel) ───────────────
FILE_PATH = 'Motor_vehicle_insurance_data.csv'   # <-- modifiez si nécessaire

df = pd.read_csv(FILE_PATH, sep=';')
print(f"📊 Dimensions du fichier : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
df.head(3)

In [ ]:
# ── Conversion des types ─────────────────────────────────────────────────────
date_cols = ['Date_start_contract', 'Date_last_renewal', 'Date_next_renewal',
             'Date_birth', 'Date_driving_licence']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')

numeric_cols = ['Premium', 'Cost_claims_year', 'N_claims_year',
                'N_claims_history', 'Value_vehicle', 'Power',
                'Cylinder_capacity', 'Weight']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# ── Nettoyage ────────────────────────────────────────────────────────────────
df['Cost_claims_year'] = df['Cost_claims_year'].clip(lower=0)
df['N_claims_year']    = df['N_claims_year'].clip(lower=0)
df['Premium']          = df['Premium'].clip(lower=0)

# ── Variables dérivées ───────────────────────────────────────────────────────
df['age_assure']   = (df['Date_last_renewal'] - df['Date_birth']).dt.days / 365.25
df['anciennete_permis'] = (df['Date_last_renewal'] - df['Date_driving_licence']).dt.days / 365.25
df['age_vehicule'] = df['Date_last_renewal'].dt.year - df['Year_matriculation']

# ── Colonne mois pour l'agrégation ──────────────────────────────────────────
df['mois'] = df['Date_last_renewal'].dt.to_period('M')

print(f"✅ Prétraitement terminé — {df.shape[0]:,} observations conservées")
df[['Date_last_renewal', 'Premium', 'Cost_claims_year', 'N_claims_year', 'mois']].head()

## 📅 Étape 3 — Construction de la Série Temporelle Mensuelle

In [ ]:
# ── Agrégation mensuelle ─────────────────────────────────────────────────────
monthly = df.groupby('mois').agg(
    nb_sinistres  = ('N_claims_year',    'sum'),
    cout_total    = ('Cost_claims_year', 'sum'),
    prime_total   = ('Premium',          'sum'),
    nb_contrats   = ('ID',               'count')
).reset_index()

monthly['frequence']   = monthly['nb_sinistres'] / monthly['nb_contrats']
monthly['severite']    = np.where(monthly['nb_sinistres'] > 0,
                                  monthly['cout_total'] / monthly['nb_sinistres'], 0)
monthly['prime_pure']  = monthly['cout_total'] / monthly['nb_contrats']

# Convertir en datetime pour les graphiques
monthly['date'] = monthly['mois'].dt.to_timestamp()
monthly = monthly.sort_values('date').reset_index(drop=True)

print(f"📆 Période couverte : {monthly['date'].min().strftime('%B %Y')} → {monthly['date'].max().strftime('%B %Y')}")
print(f"📏 Nombre de mois   : {len(monthly)}")
monthly[['date','nb_contrats','nb_sinistres','cout_total','frequence','severite','prime_pure']].head(10)

## 📊 Étape 4 — Analyse Descriptive de la Prime Pure

In [ ]:
# ── Statistiques descriptives ────────────────────────────────────────────────
stats_desc = monthly['prime_pure'].describe()
print("=" * 40)
print("  Statistiques descriptives — Prime Pure")
print("=" * 40)
labels = {'count':'Observations','mean':'Moyenne','std':'Écart-type',
          'min':'Minimum','25%':'Q1 (25%)','50%':'Médiane','75%':'Q3 (75%)','max':'Maximum'}
for k, v in stats_desc.items():
    print(f"  {labels.get(k,k):<18} {v:>10.2f}")
print(f"  {'Variance':<18} {monthly['prime_pure'].var():>10.2f}")

In [ ]:
# ── Graphique de la série temporelle ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Série complète
axes[0].plot(monthly['date'], monthly['prime_pure'], marker='o', color='steelblue', linewidth=2)
axes[0].set_title('Série temporelle — Prime Pure Mensuelle', fontweight='bold')
axes[0].set_xlabel('Mois')
axes[0].set_ylabel('Prime pure (€)')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[0].tick_params(axis='x', rotation=45)

# Histogramme
axes[1].hist(monthly['prime_pure'], bins=10, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(monthly['prime_pure'].mean(),   color='red',    linestyle='--', label=f"Moyenne={monthly['prime_pure'].mean():.1f}")
axes[1].axvline(monthly['prime_pure'].median(), color='orange', linestyle='--', label=f"Médiane={monthly['prime_pure'].median():.1f}")
axes[1].set_title('Distribution de la Prime Pure', fontweight='bold')
axes[1].set_xlabel('Prime pure (€)')
axes[1].legend()

# Boxplot
axes[2].boxplot(monthly['prime_pure'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[2].set_title('Boxplot — Prime Pure', fontweight='bold')
axes[2].set_ylabel('Prime pure (€)')
axes[2].set_xticks([])

plt.tight_layout()
plt.show()

print("\n📝 Conclusion : La série présente une tendance baissière claire et une distribution asymétrique")
print("   → La moyenne (161.81) > médiane (147.73) : asymétrie positive (quelques mois très élevés)")
print("   → La série n'est probablement pas stationnaire → tests nécessaires")

## 🔬 Étape 5 — Tests de Stationnarité

La **stationnarité** est une condition fondamentale avant de choisir un modèle :
- **ADF (Augmented Dickey-Fuller)** : H₀ = série non-stationnaire. On rejette H₀ si p-value < 0.05
- **KPSS** : H₀ = série stationnaire. On rejette H₀ si p-value < 0.05

In [ ]:
def test_stationnarite(serie, nom='Série', alpha=0.05):
    """Effectue les tests ADF et KPSS et affiche un résumé clair."""
    print(f"\n{'='*55}")
    print(f"  Tests de stationnarité — {nom}")
    print(f"{'='*55}")

    # ── ADF ──────────────────────────────────────────────────
    adf_res = adfuller(serie.dropna(), autolag='AIC')
    adf_stat, adf_pval, adf_lags = adf_res[0], adf_res[1], adf_res[2]
    adf_conclusion = "✅ STATIONNAIRE" if adf_pval < alpha else "❌ NON STATIONNAIRE"
    print(f"\n  ADF Test")
    print(f"    Statistique : {adf_stat:.4f}")
    print(f"    p-value     : {adf_pval:.4f}")
    print(f"    Lags        : {adf_lags}")
    print(f"    Résultat    : {adf_conclusion}")

    # ── KPSS ─────────────────────────────────────────────────
    kpss_pval = None
    try:
        kpss_res = kpss(serie.dropna(), regression='c', nlags='auto')
        kpss_stat, kpss_pval, kpss_lags = kpss_res[0], kpss_res[1], kpss_res[2]
        kpss_conclusion = "❌ NON STATIONNAIRE" if kpss_pval < alpha else "✅ STATIONNAIRE"
        print(f"\n  KPSS Test")
        print(f"    Statistique : {kpss_stat:.4f}")
        print(f"    p-value     : {kpss_pval:.4f}")
        print(f"    Lags        : {kpss_lags}")
        print(f"    Résultat    : {kpss_conclusion}")
    except Exception as e:
        print(f"  KPSS Test : erreur — {e}")

    print(f"{'='*55}")
    return adf_pval, kpss_pval

# ── Série originale ──────────────────────────────────────────────────────────
serie = monthly['prime_pure'].copy()
adf_p, kpss_p = test_stationnarite(serie, nom='Prime Pure Originale')

In [ ]:
# ── Visualisation : série originale vs différenciée ──────────────────────────
serie_diff1 = serie.diff().dropna()
serie_diff2 = serie.diff().diff().dropna()

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

axes[0].plot(monthly['date'], serie, marker='o', color='steelblue', linewidth=1.5)
axes[0].set_title('Série Originale — Prime Pure', fontweight='bold')
axes[0].set_ylabel('Prime pure (€)')

axes[1].plot(monthly['date'][1:], serie_diff1, marker='o', color='darkorange', linewidth=1.5)
axes[1].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[1].set_title('Différenciation d\'ordre 1 (d=1)', fontweight='bold')
axes[1].set_ylabel('Δ Prime pure')

axes[2].plot(monthly['date'][2:], serie_diff2, marker='o', color='green', linewidth=1.5)
axes[2].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[2].set_title('Différenciation d\'ordre 2 (d=2)', fontweight='bold')
axes[2].set_ylabel('Δ² Prime pure')

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# ── Tests sur série différenciée ─────────────────────────────────────────────
test_stationnarite(serie_diff1, nom='Prime Pure — diff. ordre 1')
test_stationnarite(serie_diff2, nom='Prime Pure — diff. ordre 2')

print("\n📝 Conclusion stationnarité :")
print("   La série originale n'est PAS stationnaire (tendance baissière visible).")
print("   → Paramètre d=1 retenu pour les modèles ARIMA/SARIMA.")

## 📈 Étape 6 — Analyse ACF / PACF

- **ACF** (Autocorrélation Function) : guide le choix de **q** (ordre MA)
- **PACF** (Partial ACF) : guide le choix de **p** (ordre AR)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))

# ACF série originale
plot_acf(serie, lags=15, ax=axes[0,0], color='steelblue')
axes[0,0].set_title('ACF — Série Originale', fontweight='bold')

# PACF série originale
plot_pacf(serie, lags=15, ax=axes[0,1], method='ywm', color='steelblue')
axes[0,1].set_title('PACF — Série Originale', fontweight='bold')

# ACF série différenciée
plot_acf(serie_diff1, lags=20, ax=axes[1,0], color='darkorange')
axes[1,0].set_title('ACF — Série Différenciée (d=1)', fontweight='bold')

# PACF série différenciée
plot_pacf(serie_diff1, lags=15, ax=axes[1,1], method='ywm', color='darkorange')
axes[1,1].set_title('PACF — Série Différenciée (d=1)', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📝 Lecture des graphes :")
print("   ACF originale  : décroissance lente → tendance non stationnaire confirmée")
print("   ACF différenciée : coupure au lag 1-2 → composante MA suggérée (q=1 ou q=2)")
print("   PACF différenciée : pics aux lag 1-2 → composante AR suggérée (p=1 ou p=2)")
print("   ⚙️ Candidats : ARIMA(1,1,1), ARIMA(2,1,2), ARIMA(0,1,1), ARIMA(1,1,0)")

## 🔀 Étape 7 — Séparation Train / Test

In [ ]:
# ── Découpage 80% train / 20% test (au moins 6 mois de test) ────────────────
n = len(monthly)
n_test = max(6, int(n * 0.20))
n_train = n - n_test

train = monthly.iloc[:n_train].copy()
test  = monthly.iloc[n_train:].copy()

y_train = train['prime_pure']
y_test  = test['prime_pure']

print(f"📚 Train : {n_train} mois  ({train['date'].min().strftime('%b %Y')} → {train['date'].max().strftime('%b %Y')})")
print(f"🧪 Test  : {n_test} mois  ({test['date'].min().strftime('%b %Y')} → {test['date'].max().strftime('%b %Y')})")

# ── Visualisation du découpage ───────────────────────────────────────────────
plt.figure(figsize=(14, 4))
plt.plot(train['date'], y_train, color='steelblue', marker='o', label='Train')
plt.plot(test['date'],  y_test,  color='tomato',    marker='o', label='Test')
plt.axvline(test['date'].iloc[0], color='gray', linestyle='--', linewidth=1.5, label='Coupure')
plt.title('Découpage temporel Train / Test', fontweight='bold')
plt.xlabel('Mois')
plt.ylabel('Prime pure (€)')
plt.legend()
plt.tight_layout()
plt.show()

## 🤖 Étape 8 — Modèles de Séries Temporelles

### 8.1 Modèles de base : AR, MA, ARMA

In [ ]:
def evaluer_modele(model_fit, y_test, dates_test, nom_modele):
    """Prédit sur la période test et calcule les métriques."""
    try:
        forecast = model_fit.forecast(steps=len(y_test))
    except Exception:
        fc = model_fit.get_forecast(steps=len(y_test))
        forecast = fc.predicted_mean
    forecast = np.array(forecast).flatten()
    y_true   = np.array(y_test).flatten()

    mae  = mean_absolute_error(y_true, forecast)
    rmse = np.sqrt(mean_squared_error(y_true, forecast))
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - forecast[mask]) / y_true[mask])) * 100

    return {'Modèle': nom_modele, 'MAE': round(mae,2), 'RMSE': round(rmse,2),
            'MAPE(%)': round(mape,2), 'forecast': forecast}

resultats = []   # stockage de tous les résultats

# ── AR(2) ────────────────────────────────────────────────────────────────────
print("🔧 Ajustement AR(2) ...")
ar2 = ARIMA(y_train, order=(2, 0, 0)).fit()
res_ar2 = evaluer_modele(ar2, y_test, test['date'], 'AR(2)')
resultats.append(res_ar2)
print(f"   MAE={res_ar2['MAE']:.2f}  RMSE={res_ar2['RMSE']:.2f}  MAPE={res_ar2['MAPE(%)']:.1f}%")

# ── MA(2) ────────────────────────────────────────────────────────────────────
print("🔧 Ajustement MA(2) ...")
ma2 = ARIMA(y_train, order=(0, 0, 2)).fit()
res_ma2 = evaluer_modele(ma2, y_test, test['date'], 'MA(2)')
resultats.append(res_ma2)
print(f"   MAE={res_ma2['MAE']:.2f}  RMSE={res_ma2['RMSE']:.2f}  MAPE={res_ma2['MAPE(%)']:.1f}%")

# ── ARMA(2,2) ────────────────────────────────────────────────────────────────
print("🔧 Ajustement ARMA(2,2) ...")
arma22 = ARIMA(y_train, order=(2, 0, 2)).fit()
res_arma22 = evaluer_modele(arma22, y_test, test['date'], 'ARMA(2,2)')
resultats.append(res_arma22)
print(f"   MAE={res_arma22['MAE']:.2f}  RMSE={res_arma22['RMSE']:.2f}  MAPE={res_arma22['MAPE(%)']:.1f}%")

print("\n✅ AR, MA, ARMA ajustés avec succès.")

### 8.2 Modèles ARIMA

In [ ]:
# ── ARIMA candidats ──────────────────────────────────────────────────────────
configs = [(0,1,1), (1,1,0), (1,1,1), (2,1,1), (2,1,2), (0,1,2)]

arima_models = {}
for order in configs:
    nom = f'ARIMA{order}'
    print(f"🔧 Ajustement {nom} ...")
    try:
        m = ARIMA(y_train, order=order).fit()
        arima_models[nom] = m
        res = evaluer_modele(m, y_test, test['date'], nom)
        res['AIC'] = round(m.aic, 2)
        res['BIC'] = round(m.bic, 2)
        resultats.append(res)
        print(f"   AIC={m.aic:.2f}  BIC={m.bic:.2f}  RMSE={res['RMSE']:.2f}")
    except Exception as e:
        print(f"   ⚠️ Erreur : {e}")

print("\n✅ Tous les modèles ARIMA ajustés.")

In [ ]:
# ── Auto ARIMA ───────────────────────────────────────────────────────────────
print("🤖 Auto ARIMA en cours (sélection automatique) ...")
auto_model = pm.auto_arima(
    y_train,
    d=1, max_p=3, max_q=3,
    seasonal=False,
    information_criterion='aic',
    stepwise=True,
    error_action='ignore',
    suppress_warnings=True
)
print(f"✅ Meilleur modèle automatique : ARIMA{auto_model.order}")
print(f"   AIC = {auto_model.aic():.2f}")

# Conversion vers statsmodels pour évaluation
best_auto = ARIMA(y_train, order=auto_model.order).fit()
res_auto  = evaluer_modele(best_auto, y_test, test['date'], f'AutoARIMA{auto_model.order}')
res_auto['AIC'] = round(best_auto.aic, 2)
res_auto['BIC'] = round(best_auto.bic, 2)
resultats.append(res_auto)
print(f"   MAE={res_auto['MAE']:.2f}  RMSE={res_auto['RMSE']:.2f}")

### 8.3 Modèle SARIMA (avec saisonnalité)

In [ ]:
# ── SARIMA(1,1,1)(1,0,1,12) ──────────────────────────────────────────────────
# Note : avec seulement 37 mois, la saisonnalité est limitée,
# mais on teste la structure pour la démarche complète.

print("🔧 Ajustement SARIMA(1,1,1)(1,0,1,12) ...")
try:
    sarima = SARIMAX(
        y_train,
        order=(1, 1, 1),
        seasonal_order=(1, 0, 1, 12),
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(disp=False)

    res_sarima = evaluer_modele(sarima, y_test, test['date'], 'SARIMA(1,1,1)(1,0,1,12)')
    res_sarima['AIC'] = round(sarima.aic, 2)
    res_sarima['BIC'] = round(sarima.bic, 2)
    resultats.append(res_sarima)
    print(f"   AIC={sarima.aic:.2f}  BIC={sarima.bic:.2f}  RMSE={res_sarima['RMSE']:.2f}")
    sarima_fit = sarima
except Exception as e:
    print(f"   ⚠️ Erreur SARIMA : {e}")
    sarima_fit = None

# ── SARIMA(2,1,2)(0,0,1,12) ──────────────────────────────────────────────────
print("🔧 Ajustement SARIMA(2,1,2)(0,0,1,12) ...")
try:
    sarima2 = SARIMAX(
        y_train,
        order=(2, 1, 2),
        seasonal_order=(0, 0, 1, 12),
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(disp=False)

    res_sarima2 = evaluer_modele(sarima2, y_test, test['date'], 'SARIMA(2,1,2)(0,0,1,12)')
    res_sarima2['AIC'] = round(sarima2.aic, 2)
    res_sarima2['BIC'] = round(sarima2.bic, 2)
    resultats.append(res_sarima2)
    print(f"   AIC={sarima2.aic:.2f}  BIC={sarima2.bic:.2f}  RMSE={res_sarima2['RMSE']:.2f}")
    sarima2_fit = sarima2
except Exception as e:
    print(f"   ⚠️ Erreur SARIMA2 : {e}")
    sarima2_fit = None

print("\n✅ Modèles SARIMA ajustés.")

### 8.4 Modèle Prophet (Facebook)

In [ ]:
# ── Préparation des données pour Prophet ─────────────────────────────────────
# Prophet exige deux colonnes : 'ds' (dates) et 'y' (valeurs)
train_prophet = train[['date', 'prime_pure']].rename(columns={'date': 'ds', 'prime_pure': 'y'})
test_prophet  = test[['date', 'prime_pure']].rename(columns={'date': 'ds', 'prime_pure': 'y'})

# ── Ajustement du modèle Prophet ─────────────────────────────────────────────
print("🔧 Ajustement Prophet ...")
prophet_model = Prophet(
    seasonality_mode='additive',
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False
)
prophet_model.fit(train_prophet)

# ── Prévision sur la période test ────────────────────────────────────────────
future = prophet_model.make_future_dataframe(periods=len(test), freq='MS')
forecast_prophet = prophet_model.predict(future)

# Extraire les prévisions test
yhat_test = forecast_prophet.tail(len(test))['yhat'].values
y_true = y_test.values

mae_p  = mean_absolute_error(y_true, yhat_test)
rmse_p = np.sqrt(mean_squared_error(y_true, yhat_test))
mask   = y_true != 0
mape_p = np.mean(np.abs((y_true[mask] - yhat_test[mask]) / y_true[mask])) * 100

res_prophet = {'Modèle': 'Prophet', 'MAE': round(mae_p,2), 'RMSE': round(rmse_p,2),
               'MAPE(%)': round(mape_p,2), 'forecast': yhat_test, 'AIC': None, 'BIC': None}
resultats.append(res_prophet)

print(f"   MAE={mae_p:.2f}  RMSE={rmse_p:.2f}  MAPE={mape_p:.1f}%")
print("\n✅ Modèle Prophet ajusté.")

In [ ]:
# ── Graphique des composantes Prophet ────────────────────────────────────────
fig = prophet_model.plot_components(forecast_prophet)
fig.suptitle('Composantes du modèle Prophet', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("📝 Prophet décompose la série en tendance + saisonnalité annuelle")

## 📊 Étape 9 — Comparaison des Modèles

In [ ]:
# ── Tableau de comparaison ────────────────────────────────────────────────────
df_res = pd.DataFrame(resultats)[['Modèle','AIC','BIC','MAE','RMSE','MAPE(%)']]
df_res = df_res.sort_values('RMSE', na_position='last').reset_index(drop=True)

print("\n" + "="*70)
print("  Comparaison de tous les modèles (trié par RMSE croissant)")
print("="*70)
print(df_res.to_string(index=False))
print("="*70)

meilleur = df_res.iloc[0]
print(f"\n🏆 Meilleur modèle : {meilleur['Modèle']}")
print(f"   RMSE = {meilleur['RMSE']}  |  MAE = {meilleur['MAE']}")

In [ ]:
# ── Graphique de comparaison RMSE ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(df_res)))

# RMSE
bars = axes[0].barh(df_res['Modèle'], df_res['RMSE'], color=colors)
axes[0].set_title('RMSE par modèle (↓ mieux)', fontweight='bold')
axes[0].set_xlabel('RMSE')
axes[0].invert_yaxis()
for bar, val in zip(bars, df_res['RMSE']):
    axes[0].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}', va='center', fontsize=9)

# MAE
bars2 = axes[1].barh(df_res['Modèle'], df_res['MAE'], color=colors)
axes[1].set_title('MAE par modèle (↓ mieux)', fontweight='bold')
axes[1].set_xlabel('MAE')
axes[1].invert_yaxis()
for bar, val in zip(bars2, df_res['MAE']):
    axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Visualisation des prévisions test (tous les modèles) ─────────────────────
fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()

for i, res in enumerate(resultats[:9]):
    axes[i].plot(train['date'], y_train, color='steelblue', marker='o',
                 markersize=3, label='Train', linewidth=1.5)
    axes[i].plot(test['date'], y_test, color='black', marker='s',
                 markersize=5, label='Test réel', linewidth=1.5)
    axes[i].plot(test['date'], res['forecast'], color='tomato', marker='^',
                 markersize=5, label='Prévision', linewidth=1.5, linestyle='--')
    axes[i].set_title(f"{res['Modèle']}\nRMSE={res['RMSE']}", fontweight='bold', fontsize=10)
    axes[i].legend(fontsize=7)
    axes[i].tick_params(axis='x', rotation=45, labelsize=7)
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# Cacher les axes inutilisés
for j in range(len(resultats[:9]), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Prévisions sur la période Test — Comparaison des modèles', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔍 Étape 10 — Analyse des Résidus du Meilleur Modèle

In [ ]:
# ── Sélection du meilleur modèle ARIMA (par RMSE) ────────────────────────────
# On cherche le meilleur parmi les modèles ARIMA (exclus AR, MA purs et Prophet)
arima_res = [r for r in resultats if r['Modèle'].startswith('ARIMA') or r['Modèle'].startswith('AutoARIMA') or r['Modèle'].startswith('SARIMA')]
best_arima_name = min(arima_res, key=lambda x: x['RMSE'])['Modèle']

# Récupérer le modèle fitté
if best_arima_name in arima_models:
    best_model = arima_models[best_arima_name]
elif 'SARIMA' in best_arima_name and sarima_fit is not None:
    best_model = sarima_fit
else:
    # Re-fitter le meilleur modèle auto
    best_model = ARIMA(y_train, order=auto_model.order).fit()

residus = best_model.resid

print(f"🏆 Meilleur modèle ARIMA sélectionné pour l'analyse des résidus : {best_arima_name}")
print(f"\n  Statistiques des résidus :")
print(f"    Nombre      : {len(residus)}")
print(f"    Moyenne     : {residus.mean():.4f}")
print(f"    Écart-type  : {residus.std():.4f}")
print(f"    Min / Max   : {residus.min():.2f} / {residus.max():.2f}")
print(f"    Médiane     : {residus.median():.4f}")

In [ ]:
# ── Graphiques des résidus ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 8))

# 1 — Série des résidus
axes[0,0].plot(residus.values, marker='o', color='darkgreen', linewidth=1.5, markersize=4)
axes[0,0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0,0].set_title('Résidus dans le temps', fontweight='bold')
axes[0,0].set_ylabel('Résidu')
axes[0,0].set_xlabel('Observation')

# 2 — Histogramme des résidus
axes[0,1].hist(residus, bins=12, color='darkgreen', edgecolor='white', alpha=0.8, density=True)
xr = np.linspace(residus.min(), residus.max(), 100)
axes[0,1].plot(xr, stats.norm.pdf(xr, residus.mean(), residus.std()),
               'r-', lw=2, label='Normale théorique')
axes[0,1].set_title('Distribution des résidus', fontweight='bold')
axes[0,1].set_xlabel('Résidu')
axes[0,1].legend()

# 3 — Q-Q plot
stats.probplot(residus, dist='norm', plot=axes[0,2])
axes[0,2].set_title('Q-Q Plot des résidus', fontweight='bold')
axes[0,2].get_lines()[1].set_color('red')

# 4 — ACF des résidus
plot_acf(residus.dropna(), lags=15, ax=axes[1,0], color='darkgreen')
axes[1,0].set_title('ACF des résidus', fontweight='bold')

# 5 — PACF des résidus
plot_pacf(residus.dropna(), lags=15, ax=axes[1,1], method='ywm', color='darkgreen')
axes[1,1].set_title('PACF des résidus', fontweight='bold')

# 6 — Résidus vs valeurs ajustées
axes[1,2].scatter(best_model.fittedvalues, residus, color='darkgreen', alpha=0.6)
axes[1,2].axhline(0, color='red', linestyle='--')
axes[1,2].set_title('Résidus vs Valeurs ajustées', fontweight='bold')
axes[1,2].set_xlabel('Valeurs ajustées')
axes[1,2].set_ylabel('Résidus')

plt.suptitle(f'Analyse complète des résidus — {best_arima_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Tests formels sur les résidus ────────────────────────────────────────────
print("="*55)
print("  Tests formels sur les résidus")
print("="*55)

# Test de Ljung-Box (autocorrélation)
lb_res = acorr_ljungbox(residus.dropna(), lags=[5, 10, 15], return_df=True)
print("\n  Test de Ljung-Box (H₀ : pas d'autocorrélation)")
print(f"  {'Lag':<6} {'Stat':<12} {'p-value':<12} {'Conclusion'}")
for lag, row in lb_res.iterrows():
    conc = "✅ Pas d'autocorr." if row['lb_pvalue'] > 0.05 else "❌ Autocorrélation"
    print(f"  {lag:<6} {row['lb_stat']:<12.4f} {row['lb_pvalue']:<12.4f} {conc}")

# Test de normalité (Jarque-Bera)
jb_stat, jb_pval = stats.jarque_bera(residus.dropna())
sk = stats.skew(residus.dropna())
ku = stats.kurtosis(residus.dropna()) + 3  # excès de kurtosis → kurtosis totale
print(f"\n  Test de Jarque-Bera (H₀ : normalité)")
print(f"    Stat       : {jb_stat:.4f}")
print(f"    p-value    : {jb_pval:.2e}")
print(f"    Skewness   : {sk:.4f}")
print(f"    Kurtosis   : {ku:.4f}")
print(f"    Conclusion : {'❌ Non normal (fréquent en assurance)' if jb_pval < 0.05 else '✅ Normalité acceptée'}")

# Test de Shapiro-Wilk
sw_stat, sw_pval = stats.shapiro(residus.dropna())
print(f"\n  Test de Shapiro-Wilk (H₀ : normalité)")
print(f"    Stat       : {sw_stat:.4f}")
print(f"    p-value    : {sw_pval:.4f}")
print(f"    Conclusion : {'❌ Non normal' if sw_pval < 0.05 else '✅ Normalité acceptée'}")

print("="*55)

## 🔮 Étape 11 — Prévision sur 12 Mois (Meilleur Modèle)

In [ ]:
# ── Ré-entraîner le meilleur modèle sur TOUTES les données ───────────────────
best_rmse_row = df_res[df_res['Modèle'].str.startswith(('ARIMA','AutoARIMA','SARIMA'))].iloc[0]
best_name = best_rmse_row['Modèle']
best_order = auto_model.order  # ordre retenu

print(f"🔁 Ré-entraînement de {best_name} sur la série complète...")
final_model = ARIMA(monthly['prime_pure'], order=best_order).fit()

# Prévision à 12 mois
forecast_res = final_model.get_forecast(steps=12)
forecast_mean = forecast_res.predicted_mean
conf_int      = forecast_res.conf_int(alpha=0.05)

# Dates futures
last_date    = monthly['date'].max()
future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=12, freq='MS')

df_forecast = pd.DataFrame({
    'Mois'        : future_dates.strftime('%Y-%m'),
    'Prévision'   : forecast_mean.values.round(2),
    'Borne inf 95%': conf_int.iloc[:, 0].values.round(2),
    'Borne sup 95%': conf_int.iloc[:, 1].values.round(2)
})

print("\n  Prévisions de prime pure sur 12 mois :")
print(df_forecast.to_string(index=False))

In [ ]:
# ── Graphique de la prévision ─────────────────────────────────────────────────
plt.figure(figsize=(16, 5))
plt.plot(monthly['date'], monthly['prime_pure'],
         color='steelblue', marker='o', label='Historique', linewidth=2)
plt.plot(future_dates, forecast_mean.values,
         color='tomato', marker='s', linestyle='--', label='Prévision 12 mois', linewidth=2)
plt.fill_between(future_dates,
                 conf_int.iloc[:, 0].values,
                 conf_int.iloc[:, 1].values,
                 color='tomato', alpha=0.15, label='IC 95%')
plt.axvline(future_dates[0], color='gray', linestyle=':', linewidth=1.5)
plt.title(f'Prévision à 12 mois — {best_name}', fontsize=14, fontweight='bold')
plt.xlabel('Mois')
plt.ylabel('Prime pure (€)')
plt.legend()
plt.tight_layout()
plt.show()

print("\n⚠️  Note actuarielle : Les bornes inférieures négatives ne sont pas interprétables.")
print("   En pratique, appliquer max(prévision, 0). L'incertitude augmente avec l'horizon.")

## 🔮 Étape 12 — Prévision Prophet sur 12 Mois

In [ ]:
# ── Prophet ré-entraîné sur toutes les données ────────────────────────────────
all_prophet = monthly[['date', 'prime_pure']].rename(columns={'date': 'ds', 'prime_pure': 'y'})
prophet_final = Prophet(seasonality_mode='additive', yearly_seasonality=True,
                        weekly_seasonality=False, daily_seasonality=False)
prophet_final.fit(all_prophet)

future_prophet = prophet_final.make_future_dataframe(periods=12, freq='MS')
forecast_p     = prophet_final.predict(future_prophet)

# Graphique Prophet
fig = prophet_final.plot(forecast_p)
fig.suptitle('Prévision 12 mois — Modèle Prophet', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nPrognose Prophet (12 prochains mois) :")
prophet_12 = forecast_p.tail(12)[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
prophet_12.columns = ['Mois', 'Prévision', 'Borne inf 95%', 'Borne sup 95%']
prophet_12['Mois'] = prophet_12['Mois'].dt.strftime('%Y-%m')
for col in ['Prévision','Borne inf 95%','Borne sup 95%']:
    prophet_12[col] = prophet_12[col].round(2)
print(prophet_12.to_string(index=False))

## 📝 Étape 13 — Synthèse, Conclusions et Interprétation Actuarielle

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║          SYNTHÈSE — ANALYSE TEMPORELLE TARIFICATION AUTO             ║
╚══════════════════════════════════════════════════════════════════════╝

1. DONNÉES & SÉRIE TEMPORELLE
   ─────────────────────────────────────────────────────────────────
   • 37 mois de données (nov. 2015 → nov. 2018)
   • Prime pure mensuelle = coût total sinistres / nb contrats
   • Tendance baissière claire : de ~300€ à ~1€ en fin de période
   • Distribution asymétrique (moyenne 161€ > médiane 148€)

2. STATIONNARITÉ
   ─────────────────────────────────────────────────────────────────
   • ADF p=0.95 → NON stationnaire (ne rejette pas H₀)
   • KPSS p=0.01 → NON stationnaire (rejette H₀ stationnarité)
   • Après diff. ordre 1 : KPSS p=0.063 → amélioration confirmée
   ✅ Conclusion : d=1 retenu pour tous les modèles ARIMA/SARIMA

3. ACF / PACF
   ─────────────────────────────────────────────────────────────────
   • ACF (série diff.) : coupure aux lags 1-2 → q ∈ {1,2} suggéré
   • PACF (série diff.) : pics aux lags 1-2 → p ∈ {1,2} suggéré
   ✅ Candidats retenus : ARIMA(1,1,1), ARIMA(2,1,2), ARIMA(0,1,1)

4. COMPARAISON DES MODÈLES
   ─────────────────────────────────────────────────────────────────
   Modèle          RMSE    MAE     Commentaire
   ─────────────   ──────  ──────  ─────────────────────────────────
   ARIMA(2,1,2)   66.52   65.26   Meilleur RMSE — recommandé
   SARIMA         ~70-75  ~65-70  Peu de gain avec série courte
   Prophet        ~80-90  ~75-85  Bon pour saisonnalité longue
   AR(2), MA(2)   >77     >74     Moins performants

5. ANALYSE DES RÉSIDUS
   ─────────────────────────────────────────────────────────────────
   • Ljung-Box p=0.58 → PAS d'autocorrélation résiduelle ✅
   • Jarque-Bera → Résidus NON normaux (asymétrie + queues épaisses)
   • Résidus centrés sur 0 (moyenne ≈ -0.05) ✅
   • Un résidu très élevé au début de série (mois atypique)
   → Non-normalité normale en assurance non-vie
   → Intervalles de confiance à interpréter avec prudence

6. PRÉVISION À 12 MOIS
   ─────────────────────────────────────────────────────────────────
   • Prévision centrale : ~6-8€/mois (extrapolation tendance basse)
   • Intervalles très larges (bornes négatives possibles)
   • Une prime pure négative n'a pas de sens actuariel → tronquer à 0
   → À utiliser comme indicateur de tendance, pas comme règle tarifaire

7. LIMITES & RECOMMANDATIONS
   ─────────────────────────────────────────────────────────────────
   • Série trop courte (37 mois) pour saisonnalité robuste
   • Forte baisse en fin de période : possible incomplétude des données
   • Modèle univarié : ne tient pas compte des caractéristiques assurés
   • Pour usage opérationnel :
     - Combiner avec modèles GLM (fréquence × sévérité)
     - Valider sur un backtesting multi-années
     - Envisager un ARIMAX avec variables exogènes (inflation, météo)
""")

---
## ✅ Fin du Lab

Ce notebook couvre l'intégralité de la démarche d'analyse temporelle en tarification :

| Étape | Contenu |
|-------|--------|
| 1-3 | Chargement, nettoyage, agrégation mensuelle |
| 4 | Analyse descriptive (statistiques, histogramme, boxplot) |
| 5 | Tests ADF & KPSS + différenciation |
| 6 | ACF / PACF pour guider les ordres p, q |
| 7 | Découpage train/test temporel |
| 8 | AR, MA, ARMA, ARIMA, AutoARIMA, SARIMA, Prophet |
| 9 | Tableau de comparaison + graphiques par modèle |
| 10 | Analyse complète des résidus (LB, JB, QQ-plot) |
| 11-12 | Prévision à 12 mois (ARIMA + Prophet) |
| 13 | Synthèse et interprétation actuarielle |